<a href="https://colab.research.google.com/github/safdarmarwat/colab/blob/main/RAG_finance_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf groq openai gradio

In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from openai import OpenAI
import gradio as gr

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
# Make sure 'GROQ_API_KEY' is in your Colab Secrets (🔑 icon)
# and "Notebook access" is toggled ON.
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Ensure your PDF is uploaded to the folder sidebar (📁)
# Right-click your file, 'Copy Path', and paste it here:
PDF_PATH = "/content/report.pdf"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

# ==========================================
# 2. THE RAG ENGINE (Data Processing)
# ==========================================
def initialize_system():
    if not os.path.exists(PDF_PATH):
        return None, f"❌ Error: File not found at {PDF_PATH}. Please upload your PDF."

    try:
        print("⏳ Step 1: Reading PDF...")
        loader = PyPDFLoader(PDF_PATH)
        data = loader.load()

        print("⏳ Step 2: Splitting text into chunks...")
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        chunks = text_splitter.split_documents(data)

        print("⏳ Step 3: Creating math embeddings (HuggingFace)...")
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

        print("⏳ Step 4: Building Vector Database (FAISS)...")
        vector_db = FAISS.from_documents(chunks, embeddings)

        return vector_db, "✅ System Ready!"
    except Exception as e:
        return None, f"❌ Initialization Error: {str(e)}"

# Initialize the database
vector_store, status_message = initialize_system()
print(status_message)

# ==========================================
# 3. THE CHAT LOGIC
# ==========================================
def ask_finance_bot(question):
    if vector_store is None:
        return "System not initialized. Please check the error message in the console."

    try:
        # A. Find the most relevant parts of the PDF
        docs = vector_store.similarity_search(question, k=3)
        context = "\n".join([d.page_content for d in docs])

        # B. Send the context + question to Groq
        # We use llama-3.1-8b-instant because llama3-8b-8192 is deprecated
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {
                    "role": "system",
                    "content": f"You are a professional financial analyst. Use this context to answer: {context}"
                },
                {"role": "user", "content": question}
            ]
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"⚠️ API Error: {str(e)}"

# ==========================================
# 4. THE USER INTERFACE (Gradio)
# ==========================================
if vector_store:
    app = gr.Interface(
        fn=ask_finance_bot,
        inputs=gr.Textbox(label="Ask a question about your PDF", placeholder="e.g., What was the net income?"),
        outputs=gr.Markdown(label="Financial Analysis"),
        title="🏦 Finance AI Assistant (RAG)",
        description="Ask questions about your uploaded financial document. Powered by Groq Llama 3.1."
    )
    app.launch(debug=True)

⏳ Step 1: Reading PDF...
⏳ Step 2: Splitting text into chunks...
⏳ Step 3: Creating math embeddings (HuggingFace)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Step 4: Building Vector Database (FAISS)...
✅ System Ready!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7b56ba1dd1e77a6bd7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
